In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LassoCV
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


In [4]:
expression_data = pd.read_csv("vst_degs.csv")

In [5]:
expression_data.head()

,Unnamed: 0,chp_26,chp_31,chp_34,chp_38,chp_1,chp_3,chp_4,chp_11,chp_5,...,chp_106,chp_108,chp_109,chp_110,chp_111,chp_112,chp_113,chp_114,ipf_1125,ipf_1130
0,TMED7-TICAM2,2.541027,4.593772,4.411618,4.472838,3.411323,3.938280,3.371866,3.069651,4.132273,...,3.396303,2.997156,4.753148,2.925652,1.986403,2.458367,4.443182,4.500799,11.982528,11.948790
1,SNX5,11.664378,11.702990,11.858150,11.659046,11.590148,11.707977,11.733125,11.484413,11.441497,...,11.656847,11.740276,11.855453,11.541765,11.427096,11.610635,11.740794,11.699601,13.199010,13.516239
2,H3F3AP4,3.153035,3.372185,3.238137,2.450154,2.194782,1.734829,2.775714,1.950768,2.636781,...,0.954951,1.203205,4.585984,1.203205,8.062915,1.203205,1.203205,1.203205,17.862013,15.062174
3,AC093010.3,8.824927,8.691420,9.160692,8.637415,8.637578,8.860465,9.265420,8.937777,9.035598,...,8.597248,9.226443,9.120908,7.394812,8.768886,8.094580,9.682386,9.673836,15.016835,15.234279
4,HERC3,9.972369,10.291667,10.572234,9.945113,9.723398,10.244080,10.051501,10.134745,10.332995,...,9.749555,10.776304,11.055233,9.193449,10.619217,9.284438,10.497540,10.062599,13.940517,13.785838


In [6]:
print(expression_data.shape)

(15744, 289)


In [7]:
print("First 5 columns:", expression_data.columns[:5].tolist())

First 5 columns: ['Unnamed: 0', 'chp_26', 'chp_31', 'chp_34', 'chp_38']


In [8]:
expression = expression_data.set_index('Unnamed: 0')

In [9]:
print("First 5 column names:", expression_data.columns[:5].tolist())

First 5 column names: ['Unnamed: 0', 'chp_26', 'chp_31', 'chp_34', 'chp_38']


In [10]:
print(expression_data.shape)

(15744, 289)


In [11]:
metadata = pd.read_csv("meta_final.csv")

In [12]:
print(metadata.shape)

(288, 6)


In [13]:
print(metadata['diagnosis'].value_counts())

diagnosis
ipf        103
control    103
chp         82
Name: count, dtype: int64


In [14]:
X = expression.T #need to transpose because rows should samples and column should genes

In [15]:
y = metadata['diagnosis']

In [16]:
print("X index (first 5):", X.index[:5].tolist())
print("y index (first 5):", y.index[:5].tolist())

X index (first 5): ['chp_26', 'chp_31', 'chp_34', 'chp_38', 'chp_1']
y index (first 5): [0, 1, 2, 3, 4]


In [17]:
metadata = metadata.set_index('sample_id')

In [18]:
y = metadata['diagnosis']

In [19]:
X = X.loc[y.index]

In [20]:
lab = LabelEncoder()
y_encoded = lab.fit_transform(y)

In [23]:
print("Label encoding:")
for label, code in zip(lab.classes_, range(len(lab.classes_))):
    print(f"  {label} : {code}")

Label encoding:
  chp : 0
  control : 1
  ipf : 2


In [24]:
X_scaled = StandardScaler().fit_transform(X)

In [25]:
X_scaled = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)

In [26]:
print("Scaled data shape:", X_scaled.shape)

Scaled data shape: (288, 15744)


**Feature Selection Method 01 : SVM RFE**

In [27]:
#Define SVM classifier
svm = LinearSVC(max_iter=2000, random_state=42)

In [28]:
rfe = RFE(estimator=svm,n_features_to_select=1000,step=100)

In [29]:
rfe.fit(X_scaled, y_encoded)

,estimator,LinearSVC(max...ndom_state=42)
,n_features_to_select,1000
,step,100
,verbose,0
,importance_getter,'auto'
,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'


In [30]:
selected_svm = X_scaled.columns[rfe.support_].tolist()

In [31]:
selected_svm_df = pd.DataFrame({'gene': selected_svm })

In [32]:
selected_svm_df.to_csv( "datasets/selected_svm.csv",index=False)